# Adult Income
Data Analysis - ISAE 2025/2026

In [ ]:
if "google.colab" in str(get_ipython()):
    !curl -O https://raw.githubusercontent.com/PetitMalo/BE_data_analysis/main/bootstrap.py && python3 bootstrap.py

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from src.hyper_param_search import find_dtc_hyperparams, find_svm_hyperparams
from src.utils import get_latest_hyperparams, logger

SEED = 42

In [ ]:
data_path = "./data/"
data_file = os.path.join(data_path, "adult_sample.csv")
logger.info(f"Loading data from {data_file}")

## 1 - Data import

In [ ]:
# Load data and labels
df = pd.read_csv(
    data_file,
    header=None,
    names=[
        "age",
        "workclass",
        "education",
        "education-num",
        "occupation",
        "relationship",
        "race",
        "sex",
        "capital-gain",
        "capital-loss",
        "hours-per-week",
        "native-country",
        "income",
    ],
)
df.sample(10)  # Let's take a random sample from the full data frame

## 2 - Data analysis

In [ ]:
df.info()  # quick information concerning the dataframe

In [ ]:
df = df.rename(columns=lambda x: x.strip().replace(" ", "_").lower())
numerical_features = [col for col in df.columns if df[col].dtype in ["int64", "float64"]]
categorical_features = [col for col in df.columns if col not in numerical_features]
logger.info(f"Numerical features: {numerical_features}")
logger.info(f"Categorical features: {categorical_features}")

# One-hot encode categorical features
logger.info("One-hot encoding categorical features")
encoder = OneHotEncoder(handle_unknown="ignore", drop="if_binary")
categorical_encoded = encoder.fit_transform(df[categorical_features])
categorical_cols = encoder.get_feature_names_out(categorical_features)
categorical_cols = [col.strip().lower().replace(" ", "") for col in categorical_cols]
categorical_encoded_df = pd.DataFrame(categorical_encoded.toarray(), columns=categorical_cols)
enumerated_cat_cols = zip(range(len(categorical_cols)), categorical_cols)
logger.info(f"Categorical features encoded: {list(enumerated_cat_cols)}")

# Scale numerical features
logger.info("Scaling numerical features")
scaler = StandardScaler()
numerical_scaled = scaler.fit_transform(df[numerical_features])

# Combine numerical and encoded categorical features
df_processed = pd.concat(
    [
        pd.DataFrame(numerical_scaled, columns=numerical_features).reset_index(drop=True),
        categorical_encoded_df.reset_index(drop=True),
    ],
    axis=1,
)
df_processed["income_>50k"] = df_processed["income_>50k"].apply(lambda x: int(x))
logger.info(f"Processed DataFrame from shape {df.shape}, to shape {df_processed.shape}")

In [ ]:
# Show NaNs info
df_nan = df_processed[df_processed.isna().any(axis=1)]
nan_count = df_processed.isna().sum().sum()
logger.info(f"There are {nan_count} NaN values in the processed DataFrame.")
df_processed = df_processed.dropna(axis=0)
df_nan.head()

In [ ]:
# Let's analyze one feature (sex) according to the label (income)
counts = df.groupby(["sex", "income"]).size().unstack()

# Let's display the graphic
counts.plot(kind="bar", stacked=False)
plt.title("Income repartition according to sex")
plt.ylabel("Number of person")
plt.xlabel("Sex")
plt.xticks(rotation=0)
plt.show()

In [ ]:
X = df_processed.drop(columns=["income_>50k"]).to_numpy()
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
logger.info(f"Explained variance ratio: {pca.explained_variance_ratio_}")
logger.info(f"Singular values: {pca.singular_values_}")

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df_processed["income_>50k"], cmap="coolwarm", edgecolor="k")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Transformed Data")
plt.colorbar(label="Diagnosis")
plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
X = df_processed.drop(columns=["income_>50k"]).to_numpy()
y = df_processed["income_>50k"].to_numpy(dtype=int)
X_tsne = tsne.fit_transform(X)
fig = px.scatter(x=X_tsne[:, 0], y=X_tsne[:, 1], color=y)
fig.update_layout(
    title="t-SNE visualization of Custom Classification dataset",
    xaxis_title="First t-SNE",
    yaxis_title="Second t-SNE",
)
fig.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

## 3 - Models

### 3.1 - Decision Tree Classifier

In [ ]:
search_dtc_best_params = False
if search_dtc_best_params:
    logger.info("Searching for optimal hyperparameters...")
    dtc_hyper_params = find_dtc_hyperparams(X_train, y_train, X_val, y_val)
    logger.info(f"Optimal hyperparameters found: {dtc_hyper_params}")

In [ ]:
try:
    logger.info("Best hyperparameters:")
    logger.info(dtc_hyper_params)
except NameError:
    hyper_param_path = "./models/dtc_best_hyperparams.json"
    dtc_hyper_params = get_latest_hyperparams(prefix="dtc_best_hyperparams")
    logger.info(f"Using previously found hyperparameters from: {hyper_param_path}")
    logger.info(dtc_hyper_params)

dtf = DecisionTreeClassifier(**dtc_hyper_params, random_state=SEED)
dtf.fit(X_train, y_train)
accuracy = dtf.score(X_test, y_test)
logger.info(f"Decision Tree Classifier accuracy: {accuracy:.4f}")

### 3.2 - SVM

In [ ]:
search_svm_best_params = False
if search_svm_best_params:
    logger.info("Searching for optimal hyperparameters...")
    svm_hyper_params = find_svm_hyperparams(X_train, y_train, X_val, y_val)
    logger.info(f"Optimal hyperparameters found: {svm_hyper_params}")

In [ ]:
try:
    logger.info("Best hyperparameters:")
    logger.info(svm_hyper_params)
except NameError:
    hyper_param_path = "./models/svm_best_hyperparams.json"
    svm_hyper_params = get_latest_hyperparams(prefix="svm_best_hyperparams")
    logger.info(f"Using previously found hyperparameters from: {hyper_param_path}")
    logger.info(svm_hyper_params)

dtf = SVC(**svm_hyper_params, random_state=SEED)
dtf.fit(X_train, y_train)
accuracy = dtf.score(X_test, y_test)
logger.info(f"Support Vector Machine accuracy: {accuracy:.4f}")